In [51]:
# Import packages
import numpy as np
import pandas as pd
from functions_fcsuml import *

# Load data
df = pd.read_parquet('data_exjobb_070425.parquet')
df.head()

,TransactionId,SalePrice,Lat,Lon,BuildingAge,UtilityArea,LotArea,QualityScore,CloseToBeach,EnergyPerformance,...,MunicipalityMoveInFromCountyFrac,MunicipalityMoveOutToCountyFrac,MunicipalityMoveInFrac2yrChangeMean,MunicipalityMoveOutFrac2yrChangeMean,MunicipalityMoveInFromCountyFrac2yrChangeMean,MunicipalityMoveOutToCountyFrac2yrChangeMean,MunicipalityPopulationMeanDeviation,MedianRentMunicipality,MedianRentMunicipality2yrFracChangeMean,MedianRentMunicipalityMeanDeviation
0,10006194,3200000,56.742153,16.291778,48,143,480,36,0,60.0,...,0.025551,0.021314,0.001816,0.004082,0.000930,0.003437,4.373830,1155.0,0.091030,1.123790
1,10006232,1700000,57.278909,13.646019,54,120,750,32,0,114.0,...,0.010123,0.013869,-0.007777,0.006713,-0.001110,0.002518,1.843030,999.0,0.013451,0.972005
2,10006302,3500000,58.328009,15.103299,47,169,592,30,0,NaN,...,0.034871,0.028112,-0.003267,0.003278,-0.000932,0.000765,1.738860,1046.0,0.075010,1.017735
3,10006371,11200000,57.739437,14.122176,128,250,3311,30,0,88.0,...,0.013410,0.012764,-0.001062,0.001830,0.000457,0.000039,8.857676,1059.0,0.061507,1.030384
4,10006399,591000,57.784585,16.146614,112,87,3406,28,0,101.0,...,0.006766,0.006466,-0.001024,-0.000181,0.000441,0.000045,2.279611,909.0,0.009151,0.884437


In [52]:
# Split into targets and data, and remove undesired features.
dataframe = df.copy()
targets = pd.DataFrame(dataframe['LogAdjSalePrice202006'])
to_drop = ['TransactionId', 'BaseAreaName', 'DesoArea', 'geometry', 'DistAnyCity', 'DistAnyWater', 'SalePrice', 'LogAdjSalePrice202006', 'LogSalePrice',
            'AdjSalePrice202006']
dataframe = dataframe.drop(to_drop, axis=1)

In [55]:
# Define prepare_data function
def prepare_data(dataframe_original):
    """Takes a dataframe and finds the columns with values that are non-numerical and assigns each unique non-numerical value a numerical value."""
    dataframe = dataframe_original.copy()
    # rows = np.shape(dataframe)[0]
    cols = np.shape(dataframe)[1]
    # print(type(dataframe))
    # print(cols, rows)

    for col in range(cols):
        c = type(dataframe.iloc[0, col]) # Value of the first element in column i row 1
        # print(c)
        if c != np.float64 and c != np.int64:
            a = dataframe.iloc[:, col].unique()   # Gives the unique values of a given column
            number_of_unique_values = np.shape(a)[0]
            b = np.linspace(1, number_of_unique_values, number_of_unique_values) / number_of_unique_values
            # a.reshape(len(a),1)
            # print(np.shape(a), a, "a[0]", a[0], type(a[0]))
            
            changed_data = pd.DataFrame(dataframe[f'{dataframe.axes[1][col]}'].copy())
            # print("changed_data", changed_data, " dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            # print("changed data", type(changed_data))
            print("col",col)
            for j in range(number_of_unique_values):
                # print("col",col)
                # print(dataframe[f'{dataframe.axes[1][col]}'])
                # changed_data = dataframe[f'{dataframe.axes[1][col]}'].copy()
                
                # print(j, "type", type(a[j]), type(b[j]))
                # print("value", a[j], b[j])
                if a[j] == None:
                    changed_data = changed_data.replace('None', b[j])
                else:
                    changed_data = changed_data.replace(a[j], b[j], regex=False)
                    
                    
            # print("changed data", changed_data)
            dataframe[f'{dataframe.axes[1][col]}'] = changed_data.astype(float)
            # print("dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            
            

    return dataframe.fillna(0) # Returns the changed data with NaN values changed to zero.

In [57]:
dataframe['EstimatedContractDate'] = dataframe['EstimatedContractDate'].astype('int64') // 10**9
dataframe['YearMonth'] = dataframe['YearMonth'].astype('int64') // 10**9

prepared_data = prepare_data(dataframe)
prepared_data.insert(0, "TransactionId", df['TransactionId'])
print(prepared_data.shape)
print(prepared_data)

col 8
col 20
col 21
col 22
col 24
col 27
col 28
col 29
col 30
col 31
col 32
col 40
col 41
col 143
(91630, 165)
       TransactionId        Lat        Lon  BuildingAge  UtilityArea  LotArea  \
0           10006194  56.742153  16.291778           48          143      480   
1           10006232  57.278909  13.646019           54          120      750   
2           10006302  58.328009  15.103299           47          169      592   
3           10006371  57.739437  14.122176          128          250     3311   
4           10006399  57.784585  16.146614          112           87     3406   
...              ...        ...        ...          ...          ...      ...   
91675        9999750  58.659566  16.003465           82          164     1642   
91676        9999797  57.587832  18.712390           88          108     1770   
91677        9999865  57.057038  15.046509           33          116     1020   
91678        9999939  57.640386  18.324891           51          169      888  

In [58]:
# Chaning to numpy array
def dataframe_to_numpy(dataframe):
    data = dataframe.to_numpy()

    data, removed_indices = remove_non_numbers(data)
    print("Removed indices:", removed_indices)



    data = data.astype(float) # Changing from type object to float so that numpy functions work properly.
    return data

Transaction_ID = df.iloc[:, 0]
# Transaction_ID = Transaction_ID.to_numpy()
# Transaction_ID = Transaction_ID.astype(np.float64)
# print(f'ID0: {Transaction_ID}, type: ')
# df = df.drop("TransactionId", axis=1)

data = dataframe_to_numpy(prepared_data)
# targets = df2.values# dataframe_to_numpy(df2)
# targets = targets.values # .reshape(np.shape(targets)[0], 1)


# Divide the data set into training and testing data

training_data, testing_data, training_targets, testing_targets = train_test_split(data, targets, test_size=0.2, random_state=42)

testing_data, validation_data, testing_targets, validation_targets = train_test_split(testing_data, testing_targets, test_size=0.5, random_state=42)

print('X1 shape: ', training_data.shape)
print('X2 shape: ', testing_data.shape)
print('X3 shape: ', validation_data.shape)
print('X4 shape: ', training_targets.shape)
print('X5 shape: ', testing_targets.shape)
print('X6 shape: ', validation_targets.shape)

# datapoints = 5000
# training_data, testing_data, training_targets, testing_targets, validation_data, validation_targets = training_data[0:datapoints, :], testing_data[0:datapoints, :], training_targets[0:datapoints, :], testing_targets[0:datapoints, :], validation_data[0:datapoints, :], validation_targets[0:datapoints, :]

Removed indices: {0: 0}
X1 shape:  (73304, 164)
X2 shape:  (9163, 164)
X3 shape:  (9163, 164)
X4 shape:  (73304, 1)
X5 shape:  (9163, 1)
X6 shape:  (9163, 1)


In [59]:
# Normalizing the data
norm_data, normalization_variables = normalize(training_data)
norm_targets, max_targets, min_targets = normalize_feature(training_targets)

norm_testing_targets = (testing_targets-min_targets)/(max_targets-min_targets)
norm_validation_targets = (validation_targets-min_targets)/(max_targets-min_targets)
norm_validation_data = (validation_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
norm_testing_data = (testing_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
print(np.shape(norm_data))

(73304, 164)


c:\Users\Projektlotsarna\Code\Exjobb\Finding-comparable-sales-using-macine-learning\functions_fcsuml.py:28: RuntimeWarning: invalid value encountered in divide
  return (feature-np.min(feature)) / (np.max(feature)-np.min(feature)), np.max(feature), np.min(feature)
C:\Users\Projektlotsarna\AppData\Local\Temp\ipykernel_33624\1173231302.py:7: RuntimeWarning: invalid value encountered in divide
  norm_validation_data = (validation_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
C:\Users\Projektlotsarna\AppData\Local\Temp\ipykernel_33624\1173231302.py:8: RuntimeWarning: invalid value encountered in divide
  norm_testing_data = (testing_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])


In [60]:
version = '260326'

name_list = [f'norm_data_{version}.parquet', f'norm_targets_{version}.parquet', f'norm_testing_targets_{version}.parquet', f'norm_validation_targets_{version}.parquet',
             f'norm_validation_data_{version}.parquet', f'norm_testing_data_{version}.parquet']

data_list = [norm_data, norm_targets, norm_testing_targets, norm_validation_targets, norm_validation_data, norm_testing_data]

for i in range(len(name_list)):
    temp = pd.DataFrame(data_list[i])
    temp.to_parquet(name_list[i])
